In [ ]:
import os
from typing import cast

import numpy as np
import trimesh

from tabletop_py.utils.mesh import (
    export_geometry,
    load_geometry,
    transform_geometry,
    visualize_geometry,
)

In [ ]:
rig_mesh_dir = "/root/ws/src/tabletop/src/tabletop_moveit_config/meshes/rig"
object_mesh_dir = (
    "/root/ws/src/tabletop/src/tabletop_moveit_config/meshes/objects"
)

In [ ]:
rig_mesh_path = os.path.join(rig_mesh_dir, "rig_mod.dae")
scene = load_geometry(rig_mesh_path)
visualize_geometry(scene, notebook=True)

# Trimesh Modification

In [ ]:
mesh_filename = "simple_object.dae"
mesh_path = os.path.join(object_mesh_dir, mesh_filename)
mesh = load_geometry(mesh_path, scale=0.001)
print(mesh.extents)
visualize_geometry(mesh, notebook=True)

In [ ]:
mesh_mod = transform_geometry(mesh, {"rpy": [0.0, 0.0, 1.5707]})
print(mesh_mod.extents)
visualize_geometry(mesh_mod, notebook=True)

In [22]:
mesh_mod_path = os.path.join(
    object_mesh_dir, os.path.splitext(mesh_filename)[0] + "_mod.dae"
)
export_geometry(mesh_mod, mesh_mod_path)

In [ ]:
mesh_mod = load_geometry(mesh_mod_path)
visualize_geometry(mesh_mod, notebook=True)

# Scene modification

In [2]:
def decompose_scene(
    scene: trimesh.Scene,
) -> tuple[list[str], list[trimesh.Trimesh], np.ndarray]:
    ids: list[str] = []
    meshes: list[trimesh.Trimesh] = []
    for id, mesh in scene.geometry.items():
        ids.append(id)
        meshes.append(mesh)
    extents = np.array(list(map(lambda x: x.extents, meshes)))
    return ids, meshes, extents

In [3]:
mesh_filename = os.path.join(rig_mesh_dir, "rig_full_test.dae")
scene = cast(trimesh.Scene, load_geometry(mesh_filename))
print(f"Number of components: {len(scene.geometry)}")
# scene = scene.convert_units("meters")
scene = transform_geometry(scene, {"rpy": [-1.5707, 0.0, 0.0]})
scene = trimesh.Scene(scene.dump())

In [ ]:
ids, meshes, extents = decompose_scene(scene)
print(f"Scene extents: {scene.extents}")
print(f"Min extent: {extents.min()}, Max extent: {extents.max()}")
condition = np.logical_or(
    np.isclose(extents[:, 0], 0.9135, atol=0.05),
    np.isclose(extents[:, 0], 1.5241, atol=0.05),
)
indices = np.where(condition)[0]
print("Removing meshes at indices: ", indices)
for i in indices:
    scene.delete_geometry(ids[i])

In [ ]:
export_geometry(scene, os.path.join(mesh_dir, "rig_mod.dae"))
print("Final scene:")
visualize_geometry(scene)